# AI Assignment — Classical AI Reasoning System
### Student Roll Number: **M26CSA013**
### Academic Assignment 2026 — End-to-End Deductive Inference Pipeline

---

## 1. Assignment Objective and Requirements
The primary goal of this assignment is to develop an end-to-end, general-purpose classical AI reasoning system that:
1. Accepts a natural-language knowledge base (comprising facts, rules, and causal relations) alongside a query.
2. Translates the inputs into a formal internal logical representation (Horn clauses, First-Order Logic, and Structural Causal Models).
3. Evaluates entailment strictly using a **classical reasoning engine** (Agenda-based Forward Chaining, Goal-directed Backward Chaining, Robinson's Unification, and Pearl's SCM Counterfactual Engine).
4. Produces binary predictions (`True` / `False`) under the Closed World Assumption (CWA) and Negation-as-Failure (NAF).

---
## 2. Environment Setup and Imports
In this section, we verify the environment, configure logging, and import necessary Python standard library and third-party modules.

In [1]:
import sys
import os
import time
import copy
import re
import json
import statistics
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Set
from collections import Counter, defaultdict

# Ensure project root is in system path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"Python version: {sys.version}")
print("Student Roll Number: M26CSA013")
print("Core dependencies imported successfully.")

Python version: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
Student Roll Number: M26CSA013
Core dependencies imported successfully.


---
## 3. Configuration and Random Seeds
Setting deterministic configurations, constants, recursion depth limits, and random seeds to guarantee reproducibility.

In [2]:
import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

CONFIG = {
    "student_roll_number": "M26CSA013",
    "random_seed": RANDOM_SEED,
    "max_bc_depth": 50,
    "max_fc_iterations": 200,
    "use_forward": True,
    "use_backward": True,
    "submission_csv_path": "submission/submission.csv"
}

print("Active Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Active Configuration:
  student_roll_number: M26CSA013
  random_seed: 42
  max_bc_depth: 50
  max_fc_iterations: 200
  use_forward: True
  use_backward: True
  submission_csv_path: submission/submission.csv


---
## 4. Dataset Discovery and Loading
We investigate and load the official Hugging Face benchmark datasets:
- **RuleTaker** (`tasksource/ruletaker`): Theories requiring multi-step rule chaining.
- **FOLIO** (`tasksource/folio`): First-order logic reasoning with formal annotations.
- **CounterBench** (`CounterBench/CounterBench`): 1.2K counterfactual reasoning questions loaded individually from `data_balanced_alpha_V1.json` and `data_balanced_backdoor_V2.json`.

In [3]:
from datasets import load_dataset
from src.dataset_loader import load_counterbench

print("1. Loading RuleTaker sample (tasksource/ruletaker)...")
try:
    rt_dataset = load_dataset("tasksource/ruletaker")
    print(f"   RuleTaker loaded successfully! Splits: {list(rt_dataset.keys())}")
    print(f"   Train size: {len(rt_dataset['train'])}, Dev size: {len(rt_dataset['dev'])}, Test size: {len(rt_dataset['test'])}")
except Exception as e:
    print(f"   RuleTaker load warning: {e}")
    rt_dataset = None

print("\n2. Loading FOLIO sample (tasksource/folio)...")
try:
    folio_dataset = load_dataset("tasksource/folio")
    print(f"   FOLIO loaded successfully! Splits: {list(folio_dataset.keys())}")
    print(f"   Train size: {len(folio_dataset['train'])}, Validation size: {len(folio_dataset['validation'])}")
except Exception as e:
    print(f"   FOLIO load warning: {e}")
    folio_dataset = None

print("\n3. Loading CounterBench sample (CounterBench/CounterBench)...")
try:
    cb_splits = load_counterbench(max_examples=10)
    print(f"   CounterBench loaded successfully! Splits: {list(cb_splits.keys())}")
    for s_name, s_exs in cb_splits.items():
        print(f"   Split '{s_name}': {len(s_exs)} samples loaded")
except Exception as e:
    print(f"   CounterBench load warning: {e}")
    cb_splits = {}

1. Loading RuleTaker sample (tasksource/ruletaker)...
   RuleTaker loaded successfully! Splits: ['train', 'test', 'dev']
   Train size: 480152, Dev size: 75872, Test size: 151911

2. Loading FOLIO sample (tasksource/folio)...
   FOLIO loaded successfully! Splits: ['train', 'validation']
   Train size: 1001, Validation size: 203

3. Loading CounterBench sample (CounterBench/CounterBench)...
Loading CounterBench from 'CounterBench/CounterBench' ...
  Downloaded data_balanced_alpha_V1.json -> C:\Users\Harsh Mehta\.cache\huggingface\hub\datasets--CounterBench--CounterBench\snapshots\c6225dfa00b7a89ccfeae67797b9a6d80e48e44c\data_balanced_alpha_V1.json
  [counterbench] alpha: 10 examples | label dist: {False: 5, True: 5}
  Downloaded data_balanced_backdoor_V2.json -> C:\Users\Harsh Mehta\.cache\huggingface\hub\datasets--CounterBench--CounterBench\snapshots\c6225dfa00b7a89ccfeae67797b9a6d80e48e44c\data_balanced_backdoor_V2.json
  [counterbench] backdoor: 10 examples | label dist: {True: 5, Fa

---
## 5. Dataset Schema Inspection
Examining raw field definitions, features, and sample instances across RuleTaker, FOLIO, and CounterBench.

In [4]:
if rt_dataset:
    print("=== RuleTaker Schema & Sample ===")
    sample_rt = rt_dataset['dev'][0]
    for k, v in sample_rt.items():
        print(f"  {k}: {repr(v)[:100]}")

if folio_dataset:
    print("\n=== FOLIO Schema & Sample ===")
    sample_fl = folio_dataset['validation'][0]
    for k, v in sample_fl.items():
        print(f"  {k}: {repr(v)[:100]}")

if cb_splits and 'alpha' in cb_splits and cb_splits['alpha']:
    print("\n=== CounterBench Schema & Sample ===")
    sample_cb = cb_splits['alpha'][0]
    print(f"  ID: {sample_cb.id}")
    print(f"  Given info: {sample_cb.context}")
    print(f"  Question: {sample_cb.query}")
    print(f"  Label: {sample_cb.label}")
    print(f"  Raw keys: {list(sample_cb.raw.keys())}")

=== RuleTaker Schema & Sample ===
  context: 'The cat chases the cow. The cat is big. The cat is red. The cat is round. The cat is young. The cat
  question: 'The cow likes the tiger.'
  label: 'entailment'
  config: 'depth-1'

=== FOLIO Schema & Sample ===
  story_id: 380
  premises: 'People in this club who perform in school talent shows often attend and are very engaged with schoo
  premises-FOL: '∀x (InThisClub(x) ∧ PerformOftenIn(x, schoolTalentShow) → Attend(x, schoolEvent) ∧ VeryEngagedWith(
  conclusion: 'Bonnie performs in school talent shows often.'
  conclusion-FOL: 'InThisClub(bonnie) ∧ (Perform(bonnie, schoolTalentShow))'
  label: 'Uncertain'
  example_id: 1014

=== CounterBench Schema & Sample ===
  ID: cb_alpha_0
  Given info: We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.
  Question: Would Lumbo occur if not Ziklo instead of Ziklo?
  Label: False
  Raw keys: ['question', 'given_info', 'answer', 'type', 'question_id', 'meta']

---
## 6. Example Normalization
We construct a unified `Example` data structure to normalize heterogeneous dataset formats into a consistent schema with normalized boolean labels.

In [5]:
@dataclass
class Example:
    id: str
    context: str
    query: str
    label: Optional[bool]
    dataset: str
    split: str
    raw: dict = field(default_factory=dict)

def normalize_label(raw_label) -> Optional[bool]:
    if isinstance(raw_label, bool):
        return raw_label
    if isinstance(raw_label, str):
        val = raw_label.strip().lower()
        if val in ('true', 'entailment', '1', 'yes'):
            return True
        if val in ('false', 'not entailment', 'contradiction', '0', 'no'):
            return False
        if val in ('uncertain', 'unknown'):
            return None
    return None

print("Label normalisation test:")
for test_val in ['entailment', 'not entailment', 'True', 'False', 'Uncertain', 'yes', 'no']:
    print(f"  {test_val!r} -> {normalize_label(test_val)}")

Label normalisation test:
  'entailment' -> True
  'not entailment' -> False
  'True' -> True
  'False' -> False
  'Uncertain' -> None
  'yes' -> True
  'no' -> False


---
## 7. Logical Representation
Defining core data structures:
- `Literal`: Atomic formulas with predicate functor, arguments, and negation.
- `Clause`: Horn clauses of the form `Head :- Body_1, ..., Body_k`.
- `KnowledgeBase`: Container with (predicate, arity) hash indexing for O(1) matching.

In [6]:
def is_variable(term: str) -> bool:
    return bool(term and (term[0].isupper() or term.startswith('?')))

def is_constant(term: str) -> bool:
    return not is_variable(term)

def normalize_predicate(name: str) -> str:
    return name.strip().lower()

@dataclass(frozen=True)
class Literal:
    predicate: str
    args: tuple[str, ...]
    negated: bool = False

    def __post_init__(self):
        object.__setattr__(self, 'args', tuple(self.args))
        object.__setattr__(self, 'predicate', normalize_predicate(self.predicate))

    @property
    def arity(self) -> int:
        return len(self.args)

    def is_ground(self) -> bool:
        return all(is_constant(a) for a in self.args)

    def apply_substitution(self, subst: dict[str, str]) -> 'Literal':
        new_args = tuple(subst.get(a, a) if is_variable(a) else a for a in self.args)
        return Literal(self.predicate, new_args, self.negated)

    def __repr__(self) -> str:
        sign = "¬" if self.negated else ""
        return f"{sign}{self.predicate}({', '.join(self.args)})"

@dataclass
class Clause:
    head: Optional[Literal]
    body: list[Literal] = field(default_factory=list)
    label: str = ""

    @property
    def is_fact(self) -> bool:
        return len(self.body) == 0 and self.head is not None

    @property
    def is_rule(self) -> bool:
        return len(self.body) > 0 and self.head is not None

    def __repr__(self) -> str:
        if self.is_fact:
            return f"{self.head}."
        return f"{self.head} :- {' ∧ '.join(str(b) for b in self.body)}."

@dataclass
class KnowledgeBase:
    facts: list[Literal] = field(default_factory=list)
    rules: list[Clause] = field(default_factory=list)
    _fact_index: dict = field(default_factory=dict)
    _rule_index: dict = field(default_factory=dict)
    _fact_set: set = field(default_factory=set)

    def add_fact(self, literal: Literal) -> bool:
        key = (literal.predicate, literal.arity, literal.args, literal.negated)
        if key in self._fact_set:
            return False
        self._fact_set.add(key)
        self.facts.append(literal)
        idx_key = (literal.predicate, literal.arity)
        self._fact_index.setdefault(idx_key, []).append(literal)
        return True

    def add_rule(self, clause: Clause) -> None:
        self.rules.append(clause)
        if clause.head:
            idx_key = (clause.head.predicate, clause.head.arity)
            self._rule_index.setdefault(idx_key, []).append(clause)

    def get_facts(self, pred: str, arity: int) -> list[Literal]:
        return self._fact_index.get((normalize_predicate(pred), arity), [])

    def get_rules(self, pred: str, arity: int) -> list[Clause]:
        return self._rule_index.get((normalize_predicate(pred), arity), [])

    def contains_fact(self, literal: Literal) -> bool:
        key = (literal.predicate, literal.arity, literal.args, literal.negated)
        return key in self._fact_set

kb = KnowledgeBase()
kb.add_fact(Literal("young", ("bob",)))
kb.add_fact(Literal("kind", ("bob",)))
kb.add_rule(Clause(Literal("smart", ("X",)), [Literal("young", ("X",))]))
print(f"Sample KB: {len(kb.facts)} facts, {len(kb.rules)} rules.")

Sample KB: 2 facts, 1 rules.


---
## 8. Natural-Language and Formal Logic Parsing
Translating natural-language sentences and formal logic expressions into the internal logic representation.

In [7]:
from src.parser import parse_context, parse_query, parse_example

sample_context = "Bob is young. Bob is kind. All young people are smart. If someone is smart and kind, they are successful."
sample_query = "Bob is successful."

pr = parse_context(sample_context)
q_lit = parse_query(sample_query)
sample_kb = pr.to_kb()

print("Parsed Knowledge Base:")
print(f"Facts ({len(sample_kb.facts)}): {[str(f) for f in sample_kb.facts]}")
print(f"Rules ({len(sample_kb.rules)}): {[str(r) for r in sample_kb.rules]}")
print(f"Query Literal: {q_lit}")

Parsed Knowledge Base:
Facts (2): ['young(bob)', 'kind(bob)']
Rules (2): ['smart(V) :- young_people(V).', 'successful(V) :- smart(V) ∧ kind(V).']
Query Literal: successful(bob)


---
## 9. Unification
Robinson's First-Order Unification algorithm with variable renaming (standardisation apart).

In [8]:
from src.unification import unify_literals, rename_variables

lit_rule_head = Literal("grandparent", ("X", "Z"))
lit_goal = Literal("grandparent", ("alice", "carol"))

mgu = unify_literals(lit_rule_head, lit_goal)
print(f"Unifying {lit_rule_head} with {lit_goal}:")
print(f"  MGU substitution: {mgu}")

# Renaming variables
renamed_head, renamed_body = rename_variables(
    lit_rule_head,
    [Literal("parent", ("X", "Y")), Literal("parent", ("Y", "Z"))],
    suffix=1
)
print(f"Renamed rule: {renamed_head} :- {renamed_body}")

Unifying grandparent(X, Z) with grandparent(alice, carol):
  MGU substitution: {'X': 'alice', 'Z': 'carol'}
Renamed rule: grandparent(X_1, Z_1) :- [parent(X_1, Y_1), parent(Y_1, Z_1)]


---
## 10. Forward Chaining
Worklist/Agenda-based forward chaining algorithm iteratively deriving new facts until fixpoint.

In [9]:
from src.forward_chaining import forward_chain

kb_fc = copy.deepcopy(sample_kb)
entailed, trace, elapsed = forward_chain(kb_fc, q_lit, verbose=False)
print(f"Forward Chaining Result: Entailed={entailed} in {elapsed*1000:.3f} ms")
print("Proof Trace:")
for step in trace[:6]:
    print(f"  {step}")

Forward Chaining Result: Entailed=False in 0.026 ms
Proof Trace:
  Forward Chaining — Query: successful(bob)
  Initial facts (2): ['young(bob)', 'kind(bob)']
  
--- Iteration 1 ---
  ✗ Query successful(bob) not derivable (fixpoint after 1 iterations) → False


---
## 11. Backward Chaining
Goal-directed SLD backward chaining with cycle detection and depth bounding.

In [10]:
from src.backward_chaining import backward_chain

kb_bc = copy.deepcopy(sample_kb)
entailed_bc, trace_bc, elapsed_bc = backward_chain(kb_bc, q_lit, verbose=False)
print(f"Backward Chaining Result: Entailed={entailed_bc} in {elapsed_bc*1000:.3f} ms")
print("Trace highlights:")
for step in trace_bc[:8]:
    print(f"  {step}")

Backward Chaining Result: Entailed=False in 0.057 ms
Trace highlights:
  Backward Chaining — Query: successful(bob)
  KB: 2 facts, 2 rules
  Goal: successful(bob)
  ↓ Rule: successful(V) :- smart(V) ∧ kind(V).  [renamed suffix=1]
    Head unified: successful(V_1) → subst {'V_1': 'bob'}
    Goal: smart(bob)
    ↓ Rule: smart(V) :- young_people(V).  [renamed suffix=2]
      Head unified: smart(V_2) → subst {'V_1': 'bob', 'V_2': 'bob'}


---
## 12. Query Prediction and Counterfactual Engine
The unified `ReasoningEngine` orchestrates backward chaining, forward chaining, and counterfactual SCM inference.

In [11]:
from src.reasoning_engine import ReasoningEngine

engine = ReasoningEngine(use_forward=True, use_backward=True)
res = engine.predict(sample_context, sample_query, example_id="demo_1")
print("First-Order Logic Prediction:")
print(f"  {res.summary()}")

# Test on CounterBench Counterfactual example
cb_ctx = "We know that Ziklo causes not Blaf, Blaf causes Lumbo."
cb_q = "Would Lumbo occur if not Ziklo instead of Ziklo?"
res_cb = engine.predict(cb_ctx, cb_q, example_id="demo_cb")
print("\nCounterfactual SCM Prediction:")
print(f"  {res_cb.summary()}")
print(f"  Prediction: {res_cb.prediction}")

First-Order Logic Prediction:
  Prediction: False | Method: not_entailed_cwa | Parse: 0.08ms | BC: 0.04ms | FC: 0.01ms | Total: 0.29ms | Derived: 0 facts

Counterfactual SCM Prediction:
  Prediction: True | Method: counterfactual_scm | Parse: 0.00ms | BC: 0.00ms | FC: 0.37ms | Total: 1.23ms | Derived: 0 facts
  Prediction: True


---
## 13. Unit Tests
Executing the complete 23-test automated suite covering base facts, multi-rule chains, two premises, NAF, cycle detection, counterfactual interventions, and malformed inputs.

In [12]:
from tests.test_reasoning_engine import run_all_tests

passed, failed = run_all_tests()
print(f"Test Suite Execution Complete: {passed} Passed, {failed} Failed.")
assert failed == 0, "All unit tests must pass."


REASONING ENGINE TEST SUITE
  PASS T01 Query is fact
  PASS T02 One rule
  PASS T03 Multi-rule chain
  PASS T04 Two premises
  PASS T05 Variable substitution (FC)
  PASS T06 Repeated variables
  PASS T07 Unprovable query
  PASS T08 Cycle terminates
  PASS T09 Conflicting substitution
  PASS T10 Wrong predicate
  PASS T10b Wrong arity
  PASS T11 Duplicate facts
  PASS T12 NAF negation
  PASS T13 Grandparent example
  PASS T14 Unknown entity
  PASS T15 Empty KB
  PASS T16 Malformed input
  PASS T17 Assignment example (BC)
  PASS T18 Unify var->constant
  PASS T19 Unify var->var
  PASS T20 Assignment example (FC)
  PASS T21 Counterfactual basic chain
  PASS T22 Counterfactual negation reversal

Results: 23 passed, 0 failed out of 23 tests

Test Suite Execution Complete: 23 Passed, 0 Failed.


Could not parse query: 'Is Bob smart'


---
## 14. Dataset Evaluation
Evaluating the reasoning system across public benchmark datasets: RuleTaker, CounterBench (alpha & backdoor), and FOLIO.

In [13]:
from src.dataset_loader import load_ruletaker, load_counterbench, load_folio
from src.evaluation import Evaluator

evaluator = Evaluator(engine, verbose=False)
all_eval_records = []

# 1. RuleTaker dev split
print("Evaluating on RuleTaker (dev)...")
rt_data = load_ruletaker(max_examples=100)
if 'dev' in rt_data:
    m_rt = evaluator.evaluate(rt_data['dev'], tag="RuleTaker/dev-100")
    all_eval_records.extend(m_rt['records'])

# 2. CounterBench alpha & backdoor
print("\nEvaluating on CounterBench (alpha & backdoor)...")
cb_data = load_counterbench(max_examples=100)
if 'alpha' in cb_data:
    m_cb_a = evaluator.evaluate(cb_data['alpha'], tag="CounterBench/alpha-100")
    all_eval_records.extend(m_cb_a['records'])
if 'backdoor' in cb_data:
    m_cb_b = evaluator.evaluate(cb_data['backdoor'], tag="CounterBench/backdoor-100")
    all_eval_records.extend(m_cb_b['records'])

# 3. FOLIO validation split
print("\nEvaluating on FOLIO (validation)...")
fl_data = load_folio(max_examples=100)
if 'validation' in fl_data:
    m_fl = evaluator.evaluate(fl_data['validation'], tag="FOLIO/val-100")
    all_eval_records.extend(m_fl['records'])

Evaluating on RuleTaker (dev)...
Loading RuleTaker ...
  Loaded from: tasksource/ruletaker
  Available splits: ['train', 'test', 'dev']
  [ruletaker] train: 100 examples | label dist: {True: 51, False: 49}
  [ruletaker] test: 100 examples | label dist: {True: 53, False: 47}
  [ruletaker] dev: 100 examples | label dist: {True: 53, False: 47}
  RuleTaker loaded in 7.2s
  [1/100] [OK] id='rt_dev_0' pred=True gold=True method=backward time=0.8ms
  [2/100] [OK] id='rt_dev_1' pred=False gold=False method=not_entailed_cwa time=0.9ms
  [3/100] [OK] id='rt_dev_2' pred=True gold=True method=backward time=0.6ms
  [4/100] [OK] id='rt_dev_3' pred=False gold=False method=not_entailed_cwa time=0.9ms
  [5/100] [X] id='rt_dev_4' pred=False gold=True method=not_entailed_cwa time=0.9ms

Evaluation Report: RuleTaker/dev-100
  Total examples:   100
  Labeled examples: 100
  Correct:          57
  Accuracy:         57.00%
  Precision:        0.6000
  Recall:           0.5660
  F1 Score:         0.5825
  Par

  FOL: no body parsed from '(Chaperone(bonnie, highSchoolDance) ⊕ Perform(bonnie, schoolTalentShow))' in '(Chaperone(bonnie, highSchoolDance) ⊕ Perform(bonnie, schoolTalentShow)) → (YoungChildren(bonnie) ⊕ Teenager(bonnie)) ∧ WishToFurther(bonnie, academicCareer)) ∧ (InActive(bonnie) ∧ Disinterested(bonnie) ∧ MemberOf(bonnie, community))'
  FOL: no body parsed from '(Employee(x) ∧ Schedule(x, meeting, customers))' in '(Employee(x) ∧ Schedule(x, meeting, customers)) → AppearIn(x, company)'
  FOL: no body parsed from '(Employee(x) ∧ HasLunch(x, company))' in '(Employee(x) ∧ HasLunch(x, company)) → Schedule(x, meeting, customers)'
  FOL: no body parsed from '(Employee(x) ∧ HasLunch(x, home))' in '(Employee(x) ∧ HasLunch(x, home)) → Work(x, home)'
  FOL: no body parsed from '(Employee(x) ∧ (¬In(x, homecountry)))' in '(Employee(x) ∧ (¬In(x, homecountry))) → Work(x, home)'
  FOL: no body parsed from '(Employee(x) ∧ Schedule(x, meeting, customers))' in '(Employee(x) ∧ Schedule(x, meeting, cus

---
## 15. Comparative Experiments
Benchmarking Backward Chaining vs. Forward Chaining on identical subsets to analyze performance and execution time.

In [14]:
engine_bc_only = ReasoningEngine(use_forward=False, use_backward=True)
engine_fc_only = ReasoningEngine(use_forward=True, use_backward=False)

eval_bc = Evaluator(engine_bc_only, verbose=False)
eval_fc = Evaluator(engine_fc_only, verbose=False)

sample_eval_set = rt_data['dev'][:40] if ('dev' in rt_data and rt_data['dev']) else []

if sample_eval_set:
    print("--- Backward Chaining Only ---")
    m_bc = eval_bc.evaluate(sample_eval_set, tag="BC-Only (40)")
    print("\n--- Forward Chaining Only ---")
    m_fc = eval_fc.evaluate(sample_eval_set, tag="FC-Only (40)")
    
    print(f"\nComparative Summary:")
    print(f"  Backward Chaining: Accuracy={m_bc['accuracy']:.2f}%, Avg Time={m_bc['timing_stats']['total_per_query']['mean']*1000:.3f} ms")
    print(f"  Forward Chaining:  Accuracy={m_fc['accuracy']:.2f}%, Avg Time={m_fc['timing_stats']['total_per_query']['mean']*1000:.3f} ms")

--- Backward Chaining Only ---
  [1/40] [OK] id='rt_dev_0' pred=True gold=True method=backward time=1.0ms
  [2/40] [OK] id='rt_dev_1' pred=False gold=False method=not_entailed_cwa time=0.8ms
  [3/40] [OK] id='rt_dev_2' pred=True gold=True method=backward time=0.8ms
  [4/40] [OK] id='rt_dev_3' pred=False gold=False method=not_entailed_cwa time=0.7ms
  [5/40] [X] id='rt_dev_4' pred=False gold=True method=not_entailed_cwa time=0.7ms

Evaluation Report: BC-Only (40)
  Total examples:   40
  Labeled examples: 40
  Correct:          24
  Accuracy:         60.00%
  Precision:        0.6316
  Recall:           0.5714
  F1 Score:         0.6000
  Parse failures:   0 (0.0%)
  Confusion Matrix: [[12, 7], [9, 12]]

  Timing (per query):
    Parse:         0.259ms avg
    Backward:      0.018ms avg
    Forward:       0.000ms avg
    Total:         0.412ms avg
    Wall time:     0.02s total


--- Forward Chaining Only ---
  [1/40] [OK] id='rt_dev_0' pred=True gold=True method=forward time=0.8ms
  [2

---
## 16. Accuracy and Timing Results
Comprehensive summary of evaluation metrics and microsecond-level timing breakdowns across all datasets.

In [15]:
print("="*65)
print("COMPREHENSIVE BENCHMARK EVALUATION RESULTS")
print(f"Student Roll Number: {CONFIG['student_roll_number']}")
print("="*65)
print(f"RuleTaker (dev-100):          Accuracy={m_rt['accuracy']:.2f}% | F1={m_rt['f1']:.4f} | Time={m_rt['timing_stats']['total_per_query']['mean']*1000:.3f} ms")
print(f"CounterBench (alpha-100):    Accuracy={m_cb_a['accuracy']:.2f}% | F1={m_cb_a['f1']:.4f} | Time={m_cb_a['timing_stats']['total_per_query']['mean']*1000:.3f} ms")
print(f"CounterBench (backdoor-100): Accuracy={m_cb_b['accuracy']:.2f}% | F1={m_cb_b['f1']:.4f} | Time={m_cb_b['timing_stats']['total_per_query']['mean']*1000:.3f} ms")
print(f"FOLIO (val-100):             Accuracy={m_fl['accuracy']:.2f}% | F1={m_fl['f1']:.4f} | Time={m_fl['timing_stats']['total_per_query']['mean']*1000:.3f} ms")
print("="*65)

COMPREHENSIVE BENCHMARK EVALUATION RESULTS
Student Roll Number: M26CSA013
RuleTaker (dev-100):          Accuracy=57.00% | F1=0.5825 | Time=0.405 ms
CounterBench (alpha-100):    Accuracy=100.00% | F1=1.0000 | Time=0.052 ms
CounterBench (backdoor-100): Accuracy=65.00% | F1=0.6316 | Time=0.037 ms
FOLIO (val-100):             Accuracy=44.62% | F1=0.3571 | Time=0.296 ms


---
## 17. Reasoning Traces
Inspecting explainable step-by-step traces generated by the classical engines for First-Order Logic and Counterfactual Causal deductions.

In [16]:
print("=== TRACE 1: POSITIVE ENTAILMENT TRACE (GRANDPARENT DEDUCTION) ===")
gp_ctx = "Alice is parent of Bob. Bob is parent of Carol. If someone is parent of someone and that person is parent of another, they are grandparent of another."
gp_query = "Alice is grandparent of Carol."
res_gp = engine.predict(gp_ctx, gp_query, example_id="trace_grandparent")
print(f"Prediction: {res_gp.prediction} (Method: {res_gp.method})")
for line in res_gp.trace[:10]:
    print(f"  {line}")

print("\n=== TRACE 2: COUNTERFACTUAL CAUSAL SCM DEDUCTION ===")
cb_demo_ctx = "We know that Ziklo causes not Blaf, Blaf causes Trune, and Trune causes Lumbo."
cb_demo_q = "Would Lumbo occur if not Ziklo instead of Ziklo?"
res_cb_demo = engine.predict(cb_demo_ctx, cb_demo_q, example_id="trace_cb")
print(f"Prediction: {res_cb_demo.prediction} (Method: {res_cb_demo.method})")
for line in res_cb_demo.trace:
    print(f"  {line}")

=== TRACE 1: POSITIVE ENTAILMENT TRACE (GRANDPARENT DEDUCTION) ===
Prediction: False (Method: not_entailed_cwa)
  [trace_grandparent] Context parsed: 2 facts, 1 rules, 0 parse failures
    Query literal: grandparent_of_carol(alice)
  --- Backward Chaining ---
  Backward Chaining — Query: grandparent_of_carol(alice)
  KB: 2 facts, 1 rules
  Goal: grandparent_of_carol(alice)
  ✗ Goal grandparent_of_carol(alice) FAILED (no fact or rule applies)
  
✗ NOT ENTAILED: grandparent_of_carol(alice) has no proof (elapsed: 0.000010s)
  --- Forward Chaining ---
  Forward Chaining — Query: grandparent_of_carol(alice)

=== TRACE 2: COUNTERFACTUAL CAUSAL SCM DEDUCTION ===
Prediction: True (Method: counterfactual_scm)
  === Counterfactual Reasoning Engine (Pearl SCM) ===
  Target variable: lumbo
  Intervention: do(ziklo = False)
  Causal rule: ziklo -> ¬blaf
  Causal rule: blaf -> trune
  Causal rule: trune -> lumbo
  Step 2 (Graph Surgery): Applied interventions {'ziklo': False}
  Step 3 (Deduction): F

---
## 18. Error Analysis and Limitations
Analysis of misclassifications, open-world vs closed-world semantics, and counterfactual backdoor probabilistic limits.

In [17]:
print("Error Breakdown Analysis:")
print("1. Open-World Assumption (OWA) vs Closed-World Assumption (CWA):")
print("   - FOLIO includes 'Uncertain' instances where conclusions are consistent but not strictly entailed.")
print("   - Under CWA, our engine classifies unproved statements as False, causing false negatives.")
print("2. Stochastic Backdoor Adjustments:")
print("   - CounterBench backdoor examples contain Bernoulli distributions (Bern(0.2)).")
print("   - Deterministic constraint propagation handles structural causal paths but cannot perform Bayesian conditioning.")
print("3. Disjunctive Rule Heads:")
print("   - Statements of the form A -> B v C cannot be resolved via standard Horn-clause chaining.")
print(f"4. Parse Failures:")
print(f"   - RuleTaker parse failure rate:    {m_rt['parse_fail_rate']:.2f}%")
print(f"   - CounterBench parse failure rate: {m_cb_a['parse_fail_rate']:.2f}%")
print(f"   - FOLIO parse failure rate:        {m_fl['parse_fail_rate']:.2f}%")

Error Breakdown Analysis:
1. Open-World Assumption (OWA) vs Closed-World Assumption (CWA):
   - FOLIO includes 'Uncertain' instances where conclusions are consistent but not strictly entailed.
   - Under CWA, our engine classifies unproved statements as False, causing false negatives.
2. Stochastic Backdoor Adjustments:
   - CounterBench backdoor examples contain Bernoulli distributions (Bern(0.2)).
   - Deterministic constraint propagation handles structural causal paths but cannot perform Bayesian conditioning.
3. Disjunctive Rule Heads:
   - Statements of the form A -> B v C cannot be resolved via standard Horn-clause chaining.
4. Parse Failures:
   - RuleTaker parse failure rate:    0.00%
   - CounterBench parse failure rate: 0.00%
   - FOLIO parse failure rate:        16.00%


---
## 19. Generation of submission.csv
Programmatically generating and validating the final `submission.csv` containing `id` and `predicted_label`.

In [18]:
from src.evaluation import generate_submission_csv

output_csv = "submission/submission.csv"
generate_submission_csv(all_eval_records, output_csv)

# Schema validation
import pandas as pd
df = pd.read_csv(output_csv)
print(f"\nValidation of {output_csv}:")
print(f"  Row count: {len(df)}")
print(f"  Columns: {list(df.columns)}")
print(f"  Null count: {df.isnull().sum().to_dict()}")
print(f"  Label distribution:\n{df['predicted_label'].value_counts().to_dict()}")


[submission.csv] Written 400 rows to: submission/submission.csv
  Validation: OK | Labels present: {False, True}

Validation of submission/submission.csv:
  Row count: 400
  Columns: ['id', 'predicted_label']
  Null count: {'id': 0, 'predicted_label': 0}
  Label distribution:
{False: 230, True: 170}
